## Harmonization & Country Spine (Step 0 of the metric pass)

Builds the **country spine** — the canonical country universe every metric is scored
against — and demonstrates the **ISO3 harmonization layer** used by all metric-pass
notebooks.

**Spine rule (D2, revised 2026-07-10):** ALL economies with WDI population data,
excluding closed regimes (PRK, ERI, TKM). No population floor — thin-coverage
countries are retained and carry reliability flags downstream (option A). Non-sovereign
territories (HKG, PRI, etc.) are **included and flagged** (`is_territory`), so
downstream views can filter to sovereigns.

**Harmonization:** all country-key resolution lives in `src/country_harmonization.py`
(code overrides incl. COW/GW→ISO3, name overrides, drop-lists) — import `add_iso3()`;
never resolve country keys ad hoc in a notebook. Unresolvable tokens map to `None`
(verified legitimate exclusions: defunct states, quasi-states, aggregates).

**Output:** `data/processed/country_spine.csv` (git-tracked config).
**Inputs:** `data/processed/wdi_clean.csv` (population; run nb 09 first if stale).
**Methodology home:** `docs/metric_methodology.md` §2.

Re-run only when: WDI population is refreshed (new spine vintage), the closed-regime
list changes, or the resolver's override maps are extended.

In [1]:
# CELL 1 — bootstrap: locate src/, imports, config
import sys
from pathlib import Path

# Find src/ dynamically — works on any machine
sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / '.env').exists()) / 'src'))

from config import *
import pandas as pd
import os

# The canonical country-key resolver (built at Step-0 harmonization, 2026-07-10)
from country_harmonization import add_iso3, TERRITORIES, EXCLUDE_CLOSED, VALID

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"Resolver loaded: {len(VALID)} valid ISO3 tokens; {len(TERRITORIES)} territories; excluded: {sorted(EXCLUDE_CLOSED)}")

PROJECT_ROOT: /Users/boulanger/Documents/governance-framework
Resolver loaded: 250 valid ISO3 tokens; 21 territories; excluded: ['ERI', 'PRK', 'TKM']


In [2]:
# CELL 2 — build country spine: ALL economies with population data,
# minus closed regimes; territories flagged. (D2 as revised 2026-07-10.)
wdi = pd.read_csv(os.path.join(PROCESSED_DIR, "wdi_clean.csv"), low_memory=False)
wdi = add_iso3(wdi, filename_hint="wdi_clean.csv")

# latest population per country (derived from data — no hardcoded year)
w = wdi[wdi["iso3"].notna() & wdi["wdi_population_total"].notna()]
latest = w.sort_values("year").groupby("iso3").tail(1)

spine = latest[~latest["iso3"].isin(EXCLUDE_CLOSED)].copy()
spine["is_territory"] = spine["iso3"].isin(TERRITORIES)
spine = spine[["iso3", "country_name", "year", "wdi_population_total", "is_territory"]] \
          .rename(columns={"year": "pop_year", "wdi_population_total": "population"}) \
          .sort_values("iso3").reset_index(drop=True)

print(f"Spine: {len(spine)} economies  |  territories: {int(spine['is_territory'].sum())}")
print(f"Population range: {spine['population'].min():,.0f} — {spine['population'].max():,.0f}")
print(f"Pop years: {spine['pop_year'].min()} — {spine['pop_year'].max()}")

Spine: 213 economies  |  territories: 21
Population range: 9,492 — 1,463,865,525
Pop years: 2025 — 2025


In [3]:
# CELL 3 — write country_spine.csv (git-tracked config; the canonical scoring universe)
spine_path = os.path.join(PROCESSED_DIR, "country_spine.csv")
spine.to_csv(spine_path, index=False)
print(f"Written: {spine_path}")
print(f"Shape: {spine.shape}")
print(spine.head(3).to_string(index=False))

Written: /Users/boulanger/Documents/governance-framework/data/processed/country_spine.csv
Shape: (213, 5)
iso3 country_name  pop_year  population  is_territory
 ABW        Aruba      2025    108785.0          True
 AFG  Afghanistan      2025  43844111.0         False
 AGO       Angola      2025  39040039.0         False


In [4]:
# CELL 4 — spine integrity checks: fail loudly if any invariant breaks
from country_harmonization import VALID, EXCLUDE_CLOSED, TERRITORIES

assert spine["iso3"].is_unique, "DUPLICATE iso3 in spine"
assert spine["iso3"].isin(VALID).all(), f"INVALID iso3: {spine.loc[~spine['iso3'].isin(VALID),'iso3'].tolist()}"
assert not spine["iso3"].isin(EXCLUDE_CLOSED).any(), "Closed-regime country leaked into spine"
assert spine["population"].notna().all() and (spine["population"] > 0).all(), "Bad population values"

print("All integrity checks passed.")
print(f"{len(spine)} economies | {int(spine['is_territory'].sum())} territories flagged")
print("Territories:", ", ".join(sorted(spine.loc[spine['is_territory'], 'iso3'])))

All integrity checks passed.
213 economies | 21 territories flagged
Territories: ABW, ASM, BMU, CUW, CYM, FRO, GIB, GRL, GUM, HKG, IMN, MAC, MAF, MNP, NCL, PRI, PYF, SXM, TCA, VGB, VIR


In [5]:
# CELL 5 — per-metric coverage table: current-coverage on spine, cadence, history depth.
# Reproducible replacement for the Step-0 scratch audit; evidence base for the
# Step-0.5 parameter locks (inclusion window/threshold, D7 reliability, momentum).
import glob

SKIP = {"source_registry.csv", "country_spine.csv", "metric_coverage.csv"}
WINDOWS = [4]                          # recency window LOCKED at 4yr (Step-0.5, 2026-07-10)
SPINE = set(spine["iso3"])                                    # all 213 economies
SPINE_SOV = set(spine.loc[~spine["is_territory"], "iso3"])    # 192 sovereigns — inclusion denominator

rows = []
for path in sorted(glob.glob(os.path.join(PROCESSED_DIR, "*.csv"))):
    fname = os.path.basename(path)
    if fname in SKIP or fname.startswith("_"):
        continue
    df = pd.read_csv(path, low_memory=False)
    try:
        df = add_iso3(df, filename_hint=fname)
    except ValueError as e:
        print(f"  !! {fname}: {e} — SKIPPED (needs explicit handling)")
        continue
    d = df[df["iso3"].isin(SPINE)]
    idlike = {"iso3", "year", "assessment_year", "country_name", "country_text_id",
              "country_id", "country_code", "iso2", "country", "cow_code",
              "country_name_source"}
    ycol = "year" if "year" in d.columns else (
           "assessment_year" if "assessment_year" in d.columns else None)
    file_max_yr = int(pd.to_numeric(d[ycol], errors="coerce").max()) if ycol else None
    metrics = [c for c in d.columns if c not in idlike]

    for m in metrics:
        dd = d[d[m].notna()]
        n_c = dd["iso3"].nunique()
        row = {"file": fname, "metric": m, "spine_countries_any": n_c,
               "panel_fill_pct": round(100 * len(dd) / max(len(d), 1), 1),
               "latest_yr_file": file_max_yr}
        if ycol and len(dd):
            yrs = pd.to_numeric(dd[ycol], errors="coerce")
            latest_per = dd.assign(_y=yrs).groupby("iso3")["_y"].max()
            per_obs = dd.groupby("iso3")[ycol].nunique().mean()
            span = yrs.max() - yrs.min() + 1
            cadence = ("snapshot" if per_obs <= 1.2 else
                       "irregular" if per_obs < 0.6 * span else "annual")
            row.update({"cadence": cadence,
                        "hist_depth_avg_yrs": round(per_obs, 1),
                        "median_latest_yr": int(latest_per.median())})
            # sovereign-core denominator for the inclusion test (§4)
            dd_sov = dd[dd["iso3"].isin(SPINE_SOV)]
            latest_per_sov = dd_sov.assign(_y=pd.to_numeric(dd_sov[ycol], errors="coerce")) \
                                   .groupby("iso3")["_y"].max()
            for w in WINDOWS:
                # cadence-relative recency: annual judged vs window;
                # irregular/snapshot: latest-available = current
                if cadence == "annual":
                    cur_all = (latest_per >= file_max_yr - w).sum()
                    cur_sov = (latest_per_sov >= file_max_yr - w).sum()
                else:
                    cur_all, cur_sov = n_c, dd_sov["iso3"].nunique()
                row[f"curcov_pct_w{w}"] = round(100 * cur_all / len(SPINE), 1)          # vs 213
                row[f"curcov_sov_pct_w{w}"] = round(100 * cur_sov / len(SPINE_SOV), 1)  # vs 192 (INCLUSION)
        else:
            n_sov = dd[dd["iso3"].isin(SPINE_SOV)]["iso3"].nunique()
            row.update({"cadence": "snapshot(no-year)", "hist_depth_avg_yrs": 0,
                        "median_latest_yr": None})
            for w in WINDOWS:
                row[f"curcov_pct_w{w}"] = round(100 * n_c / len(SPINE), 1)
                row[f"curcov_sov_pct_w{w}"] = round(100 * n_sov / len(SPINE_SOV), 1)
        rows.append(row)

coverage = pd.DataFrame(rows)
print(f"Metrics catalogued: {len(coverage)}")
print(f"Annual metrics >=60% of sovereigns (w4): "
      f"{(coverage[coverage.cadence=='annual']['curcov_sov_pct_w4'] >= 60).sum()} "
      f"of {(coverage.cadence=='annual').sum()}")
print(coverage["cadence"].value_counts().to_string())

Metrics catalogued: 458
Annual metrics >=60% of sovereigns (w4): 246 of 255
cadence
annual               255
snapshot(no-year)    144
irregular             47
snapshot              12


In [6]:
# CELL 6 — write metric_coverage.csv (git-TRACKED: the auditable evidence base
# behind Step-0.5 parameter locks and Step-1 inclusion decisions)
cov_path = os.path.join(PROCESSED_DIR, "metric_coverage.csv")
coverage.sort_values(["file", "metric"]).to_csv(cov_path, index=False)
print(f"Written: {cov_path}")
print(f"Shape: {coverage.shape}")

Written: /Users/boulanger/Documents/governance-framework/data/processed/metric_coverage.csv
Shape: (458, 10)


In [7]:
# CELL 7 — per-country coverage table (present-metric count per spine country).
# Diagnostic/reference: the evidence base for D7 reliability thresholds and the
# master's coverage characterization. Regenerable — the authoritative live figures
# live here, NOT hardcoded into the docs.
import glob

ID_COLS = {"iso3", "year", "assessment_year", "country_name", "country_text_id",
           "country_id", "country_code", "iso2", "country", "cow_code",
           "country_name_source"}
SKIP_COV = {"source_registry.csv", "country_spine.csv", "metric_coverage.csv",
            "country_coverage.csv"}

counts = {i: 0 for i in spine["iso3"]}
for path in sorted(glob.glob(os.path.join(PROCESSED_DIR, "*.csv"))):
    fname = os.path.basename(path)
    if fname in SKIP_COV or fname.startswith("_"):
        continue
    df = pd.read_csv(path, low_memory=False)
    try:
        df = add_iso3(df, filename_hint=fname)
    except ValueError:
        continue
    ycol = "year" if "year" in df.columns else (
           "assessment_year" if "assessment_year" in df.columns else None)
    if ycol:                                    # latest slice per country
        df = df.sort_values(ycol).groupby("iso3").tail(1)
    mets = [c for c in df.columns if c not in ID_COLS]
    for iso in df["iso3"].dropna().unique():
        if iso in counts:
            counts[iso] += int(df[df["iso3"] == iso][mets].notna().any().sum())

cov_country = spine[["iso3", "country_name", "is_territory"]].copy()
cov_country["metrics_present"] = cov_country["iso3"].map(counts)
cov_country = cov_country.sort_values("metrics_present", ascending=False).reset_index(drop=True)

sov = cov_country[~cov_country["is_territory"]]["metrics_present"]
print(f"Sovereigns (192): min {sov.min()}, median {int(sov.median())}, max {sov.max()}")
print(f"Territories (21): min {cov_country[cov_country.is_territory]['metrics_present'].min()}, "
      f"max {cov_country[cov_country.is_territory]['metrics_present'].max()}")
print(f"All below 84 metrics are territories: "
      f"{(cov_country[cov_country.metrics_present < 84]['is_territory']).all()}")

Sovereigns (192): min 84, median 358, max 385
Territories (21): min 3, max 258
All below 84 metrics are territories: True


In [9]:
# CELL 8 — write country_coverage.csv (git-tracked reference; live figures for D7 / docs)
cc_path = os.path.join(PROCESSED_DIR, "country_coverage.csv")
cov_country.to_csv(cc_path, index=False)
print(f"Written: {cc_path}  |  shape: {cov_country.shape}")

Written: /Users/boulanger/Documents/governance-framework/data/processed/country_coverage.csv  |  shape: (213, 4)
